In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load metric data
df_metric = pd.read_csv("metric.csv")

# Ensure timestamp is localized to UTC+8 (Asia/Shanghai) as required
tz_sh = pytz.timezone('Asia/Shanghai')
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s').dt.tz_localize(tz_sh)

# Aggregate per component-KPI (global over full series)
agg_metric = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        p95=('value', lambda s: s.quantile(0.95) if s.notna().any() else float('nan')),
        min_ts=('timestamp', 'min'),
        max_ts=('timestamp', 'max'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        mean_value=('value', 'mean')
    )
)

# Prepare compact result view (only requested columns), sorted by p95 desc, top 20
result = agg_metric[['cmdb_id', 'kpi_name', 'p95', 'count']].sort_values('p95', ascending=False).head(20)

# Display the compact result (agg_metric is kept in memory for later use)
result

```
Out[1]:
```
summary = (
    "Summary of top metric series by global P95 (from the aggregated metric table):\n"
    "- Redis01 dominates the top ranks: network packets in/out and several Redis KPIs (total_commands_processed, maxmemory, keyspace_hits) have the highest P95 values (up to ~6.16e9) with point counts between 15 and 31.\n"
    "- Redis02 also shows very high Redis KPI P95s (~4.28–4.29e9, counts ~31).\n"
    "- Multiple Tomcat instances (Tomcat01–Tomcat04) have JVM memory KPIs with P95 ≈ 4.19e9 (each with 6 points).\n"
    "- A few Docker container metrics appear in the top 20 with P95 ≈ 3.07e9 (small point counts).\n"
    "The table returned contains columns: cmdb_id, kpi_name, p95, count and is sorted by p95 descending (top 20 shown)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name           p95  count
877    Redis01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  6.159854e+09     15
878    Redis01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  5.603234e+09     15
911    Redis01  redis-Redis_6379_Redis  (total_commands_proces...  5.461277e+09     31
904    Redis01                redis-Redis_6379_Redis  (maxmemory)  4.294967e+09     31
1015   Redis02                redis-Redis_6379_Redis  (maxmemory)  4.294967e+09     31
1022   Redis02  redis-Redis_6379_Redis  (total_commands_proces...  4.278665e+09     31
1381  Tomcat03           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory  4.187619e+09      6
1379  Tomcat03             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory  4.187619e+09      6
1490  Tomcat04           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory  4.187619e+09      6
1488  Tomcat04             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory  4.187619e+09      6
1270  Tomcat02           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory  4.187619e+09      6
1268  Tomcat02             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory  4.187619e+09      6
1165  Tomcat01           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory  4.187619e+09      6
1163  Tomcat01             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory  4.187619e+09      6
899    Redis01            redis-Redis_6379_Redis  (keyspace_hits)  4.163119e+09     31
1010   Redis02            redis-Redis_6379_Redis  (keyspace_hits)  3.102214e+09     31
1684  dockerB1  Container-DOCKER_CONTAINER_464dc801314b--bcou-...  3.072000e+09      2
1690  dockerB2  Container-DOCKER_CONTAINER_d27123361435--bcou-...  3.072000e+09      3
1672  dockerA1  Container-DOCKER_CONTAINER_23bdcf67c7e3--bcou-...  3.072000e+09      1
1678  dockerA2  Container-DOCKER_CONTAINER_3bf443a64876--bcou-...  3.072000e+09      2```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
import numpy as np

# Reuse / load metric data
df_metric = pd.read_csv("metric.csv")

# Ensure timestamps are in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s').dt.tz_localize(tz)

# Ensure agg_metric exists in kernel; if not, compute global p95 per component-KPI
try:
    agg_metric
except NameError:
    agg_metric = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(p95=('value', lambda s: s.quantile(0.95)))
    )

# Define UTC+8 window: March 12, 2021 12:00 to 12:30 inclusive
start = pd.Timestamp("2021-03-12 12:00:00", tz=tz)
end = pd.Timestamp("2021-03-12 12:30:00", tz=tz)

# Filter metric records to the time window and store for later use
window_metric = df_metric[(df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end)].copy()

# Merge windowed records with agg_metric to consider only known component-KPIs and bring in global p95
merged = window_metric.merge(
    agg_metric[['cmdb_id', 'kpi_name', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='inner'
)

# If merged is empty, prepare an empty final table with the requested columns
if merged.empty:
    final = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'p95', 'window_count', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window'])
else:
    # Mark anomalies using rule: value >= p95
    merged['is_anomaly'] = merged['value'] >= merged['p95']

    # Aggregate window-level stats
    grp_all = merged.groupby(['cmdb_id', 'kpi_name', 'p95'], as_index=False).agg(
        window_count=('value', 'count'),
        max_value_in_window=('value', 'max')
    )

    # Aggregate anomaly-specific stats
    grp_anom = (
        merged[merged['is_anomaly']]
        .groupby(['cmdb_id', 'kpi_name', 'p95'], as_index=False)
        .agg(
            anomaly_count=('is_anomaly', 'sum'),
            earliest_anomaly_timestamp=('timestamp', 'min')
        )
    )

    # Combine aggregates, fill missing anomaly info
    summary = grp_all.merge(grp_anom, on=['cmdb_id', 'kpi_name', 'p95'], how='left')
    summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
    # earliest_anomaly_timestamp will be NaT for no anomalies (left as-is)

    # Compute severity ratio for sorting: max_value_in_window / p95 (handle p95 == 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        summary['severity_ratio'] = np.where(
            summary['p95'] > 0,
            summary['max_value_in_window'] / summary['p95'],
            np.where(summary['max_value_in_window'] > 0, np.inf, 0)
        )

    # Sort by anomaly_count desc, then severity_ratio desc
    summary = summary.sort_values(by=['anomaly_count', 'severity_ratio'], ascending=[False, False])

    # Prepare final compact table (only requested columns) and limit to top 20
    final = summary[['cmdb_id', 'kpi_name', 'p95', 'window_count', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']].head(20)

# final is displayed for compact output; window_metric is stored for later use
final

```
Out[2]:
```
summary = (
    "Summary: No metric records for the time window 2021-03-12 12:00 to 12:30 (UTC+8) matched the previously aggregated component-KPIs. "
    "The filtered window_metric is empty, so there are no window_count values and no anomalies (by the >= p95 rule). "
    "If you expected results, please verify the metric timestamps and timezone alignment or expand the time window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, kpi_name, p95, window_count, anomaly_count, earliest_anomaly_timestamp, max_value_in_window]
Index: []```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
import numpy as np

# Load trace data
df_trace = pd.read_csv("trace.csv")

# Ensure timestamps are localized to UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s').dt.tz_localize(tz)

# Aggregate per component-trace (global over full series) and compute P95 and count
agg_trace = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        p95=('value', lambda s: s.quantile(0.95) if s.notna().any() else np.nan),
        min_ts=('timestamp', 'min'),
        max_ts=('timestamp', 'max'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        mean_value=('value', 'mean')
    )
)

# Prepare compact result view: requested columns, sorted by p95 desc, top 20
trace_top20 = agg_trace[['cmdb_id', 'trace_name', 'p95', 'count']].sort_values('p95', ascending=False).head(20)

trace_top20

```
Out[3]:
```
summary = (
    "Summary of aggregated trace series (top by global P95):\n"
    "- The highest P95 is dockerB2.trace.self.row_count = 3398.10 (count=30), followed by dockerB1.trace.self.row_count = 3368.65 (count=30).\n"
    "- Several Tomcat instances have very high self row_count P95s: Tomcat02 2564.05, Tomcat04 2539.00, Tomcat03 2493.10, Tomcat01 2453.85 (all counts = 30 or 31).\n"
    "- Other notable self row_count series: dockerA1 2213.45, dockerA2 1990.40, IG02 1553.85, IG01 1438.45, MG02 1200.30, MG01 1145.35 (all counts = 30).\n"
    "- Lower-P95 inter-service row_count traces (e.g., trace.to_MG02, trace.from_dockerB2) appear around ~214–229 with counts = 30.\n"
    "- Overall pattern: the top-ranked traces are predominantly per-component 'trace.self.row_count' metrics with consistent sample counts (mostly 30), and a second tier of inter-service row_count traces with much lower P95 values."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                     trace_name      p95  count
257  dockerB2           trace.self.row_count  3398.10     30
239  dockerB1           trace.self.row_count  3368.65     30
149  Tomcat02           trace.self.row_count  2564.05     30
185  Tomcat04           trace.self.row_count  2539.00     31
167  Tomcat03           trace.self.row_count  2493.10     30
131  Tomcat01           trace.self.row_count  2453.85     30
203  dockerA1           trace.self.row_count  2213.45     30
221  dockerA2           trace.self.row_count  1990.40     30
23       IG02           trace.self.row_count  1553.85     30
5        IG01           trace.self.row_count  1438.45     30
107      MG02           trace.self.row_count  1200.30     30
65       MG01           trace.self.row_count  1145.35     30
263  dockerB2        trace.to_MG02.row_count   228.95     30
101      MG02  trace.from_dockerB2.row_count   228.95     30
245  dockerB1        trace.to_MG02.row_count   223.50     30
98       MG02  trace.from_dockerB1.row_count   223.50     30
56       MG01  trace.from_dockerB1.row_count   217.25     30
242  dockerB1        trace.to_MG01.row_count   217.25     30
59       MG01  trace.from_dockerB2.row_count   214.50     30
260  dockerB2        trace.to_MG01.row_count   214.50     30```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
import numpy as np

# Reuse df_trace and agg_trace if they exist in the kernel; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    tz = pytz.timezone('Asia/Shanghai')
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s').dt.tz_localize(tz)

try:
    agg_trace
except NameError:
    # compute global p95 if not present (should not normally happen)
    tz = pytz.timezone('Asia/Shanghai')
    agg_trace = (
        df_trace
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(p95=('value', lambda s: s.quantile(0.95) if s.notna().any() else np.nan))
    )

# Ensure timezone variable
tz = pytz.timezone('Asia/Shanghai')

# Define incident window (inclusive) in UTC+8
start = pd.Timestamp("2021-03-12 12:00:00", tz=tz)
end = pd.Timestamp("2021-03-12 12:30:00", tz=tz)

# Filter trace records to the time window and store for later use
window_trace = df_trace[(df_trace['timestamp'] >= start) & (df_trace['timestamp'] <= end)].copy()

# Merge with agg_trace to bring in global p95 and limit to known series
merged = window_trace.merge(
    agg_trace[['cmdb_id', 'trace_name', 'p95']],
    on=['cmdb_id', 'trace_name'],
    how='inner'
)

if merged.empty:
    final = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'p95', 'window_count', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window'])
else:
    # Determine anomalies using global p95 (value >= p95). Treat NaN p95 as no anomaly.
    merged['is_anomaly'] = (~merged['p95'].isna()) & (merged['value'] >= merged['p95'])

    # Aggregate counts and max value in window
    grp_all = merged.groupby(['cmdb_id', 'trace_name', 'p95'], as_index=False).agg(
        window_count=('value', 'count'),
        max_value_in_window=('value', 'max')
    )

    # Aggregate anomaly-specific stats
    grp_anom = (
        merged[merged['is_anomaly']]
        .groupby(['cmdb_id', 'trace_name', 'p95'], as_index=False)
        .agg(
            anomaly_count=('is_anomaly', 'sum'),
            earliest_anomaly_timestamp=('timestamp', 'min')
        )
    )

    # Combine, fill missing anomaly info
    summary = grp_all.merge(grp_anom, on=['cmdb_id', 'trace_name', 'p95'], how='left')
    summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)

    # compute severity ratio for sorting (handle p95 == 0 or NaN)
    with np.errstate(divide='ignore', invalid='ignore'):
        summary['severity_ratio'] = np.where(
            summary['p95'] > 0,
            summary['max_value_in_window'] / summary['p95'],
            np.where(summary['max_value_in_window'] > 0, np.inf, 0)
        )

    # Sort by anomaly_count desc, then severity_ratio desc
    summary = summary.sort_values(by=['anomaly_count', 'severity_ratio'], ascending=[False, False])

    # Prepare final compact table and limit to top 20
    final = summary[['cmdb_id', 'trace_name', 'p95', 'window_count', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']].head(20)

# Display compact final table; window_trace saved for later use
final

```
Out[4]:
```
summary = (
    "Summary: No trace records for March 12, 2021 12:00–12:30 (UTC+8) matched the previously aggregated trace series. "
    "The filtered window_trace is empty, so there are no window_count values and no anomalies (by the >= p95 rule). "
    "If you expected results, please verify the trace timestamps and timezone alignment or expand the time window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, trace_name, p95, window_count, anomaly_count, earliest_anomaly_timestamp, max_value_in_window]
Index: []```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

# Load log data
df_log = pd.read_csv("log.csv")

# Ensure timestamps are localized to UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s').dt.tz_localize(tz)

# Aggregate per component-log (global over full series) and compute P95 and count
agg_log = (
    df_log
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        p95=('value', lambda s: s.quantile(0.95) if s.notna().any() else np.nan),
        min_ts=('timestamp', 'min'),
        max_ts=('timestamp', 'max'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        mean_value=('value', 'mean')
    )
)

# Prepare compact result view: requested columns, sorted by p95 desc, top 20
log_top20 = agg_log[['cmdb_id', 'log_name', 'p95', 'count']].sort_values('p95', ascending=False).head(20)

# Display compact result; agg_log is stored for later use
log_top20

```
Out[5]:
```
summary = (
    "Summary of aggregated log series (top by global P95):\n"
    "- Highest log row counts: apache02 and apache01 have the largest P95 for log.row_count = 740.5 (count=31).\n"
    "- Tomcat instances follow: Tomcat03 381.0, Tomcat02 380.0, Tomcat04 380.0, Tomcat01 365.5 (all counts ~30–31).\n"
    "- Mysql01 log.row_count is much lower with P95 = 56.5 (count=26).\n"
    "- Error counts: apache01 and apache02 show nonzero P95s (2.5 and 2.0 respectively); most Tomcat error_count series have P95 = 0.0.\n"
    "- GC pause maxima: Tomcat02, Tomcat04, Tomcat03, Tomcat01 have small P95 gc_pause_max values (~0.09–0.1155).\n"
    "The returned table contains columns: cmdb_id, log_name, p95, count and is sorted by p95 descending (top 20 shown)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id          log_name       p95  count
17  apache02     log.row_count  740.5000     31
15  apache01     log.row_count  740.5000     31
10  Tomcat03     log.row_count  381.0000     31
7   Tomcat02     log.row_count  380.0000     31
13  Tomcat04     log.row_count  380.0000     31
4   Tomcat01     log.row_count  365.5000     31
1    Mysql01     log.row_count   56.5000     26
14  apache01   log.error_count    2.5000     31
16  apache02   log.error_count    2.0000     31
6   Tomcat02  log.gc_pause_max    0.1155     30
12  Tomcat04  log.gc_pause_max    0.1045     18
9   Tomcat03  log.gc_pause_max    0.0910     19
3   Tomcat01  log.gc_pause_max    0.0900     19
0    Mysql01   log.error_count    0.0000     26
8   Tomcat03   log.error_count    0.0000     31
5   Tomcat02   log.error_count    0.0000     31
2   Tomcat01   log.error_count    0.0000     31
11  Tomcat04   log.error_count    0.0000     31```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing variables if present; otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
    tz = pytz.timezone('Asia/Shanghai')
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s').dt.tz_localize(tz)

try:
    agg_log
except NameError:
    # compute global p95 per series if agg_log missing
    tz = pytz.timezone('Asia/Shanghai')
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s').dt.tz_localize(tz)
    agg_log = (
        df_log
        .groupby(['cmdb_id', 'log_name'], as_index=False)
        .agg(p95=('value', lambda s: s.quantile(0.95) if s.notna().any() else np.nan))
    )

# Ensure timestamps localized to UTC+8
tz = pytz.timezone('Asia/Shanghai')
if df_log['timestamp'].dtype == 'int64' or df_log['timestamp'].dtype == 'float64' or not pd.api.types.is_datetime64tz_dtype(df_log['timestamp']):
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s').dt.tz_localize(tz)

# Define window (inclusive) in UTC+8
start = pd.Timestamp("2021-03-12 12:00:00", tz=tz)
end = pd.Timestamp("2021-03-12 12:30:00", tz=tz)

# Filter log records to the time window and store for later use
window_log = df_log[(df_log['timestamp'] >= start) & (df_log['timestamp'] <= end)].copy()

# Merge with agg_log to include only known series and bring in global p95
merged = window_log.merge(agg_log[['cmdb_id', 'log_name', 'p95']], on=['cmdb_id', 'log_name'], how='inner')

if merged.empty:
    final = pd.DataFrame(columns=['cmdb_id', 'log_name', 'p95', 'window_count', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window'])
else:
    # Mark anomalies using rule: value >= p95 (only when p95 is not NaN)
    merged['is_anomaly'] = (~merged['p95'].isna()) & (merged['value'] >= merged['p95'])

    # Aggregate window-level stats
    grp_all = merged.groupby(['cmdb_id', 'log_name', 'p95'], as_index=False).agg(
        window_count=('value', 'count'),
        max_value_in_window=('value', 'max')
    )

    # Aggregate anomaly-specific stats
    grp_anom = (
        merged[merged['is_anomaly']]
        .groupby(['cmdb_id', 'log_name', 'p95'], as_index=False)
        .agg(
            anomaly_count=('is_anomaly', 'sum'),
            earliest_anomaly_timestamp=('timestamp', 'min')
        )
    )

    # Combine aggregates, fill missing anomaly info
    summary = grp_all.merge(grp_anom, on=['cmdb_id', 'log_name', 'p95'], how='left')
    summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)

    # Compute severity ratio for sorting: max_value_in_window / p95 (handle p95 == 0 or NaN)
    with np.errstate(divide='ignore', invalid='ignore'):
        summary['severity_ratio'] = np.where(
            summary['p95'] > 0,
            summary['max_value_in_window'] / summary['p95'],
            np.where(summary['max_value_in_window'] > 0, np.inf, 0)
        )

    # Sort by anomaly_count desc, then severity_ratio desc
    summary = summary.sort_values(by=['anomaly_count', 'severity_ratio'], ascending=[False, False])

    # Prepare final compact table and limit to top 20
    final = summary[['cmdb_id', 'log_name', 'p95', 'window_count', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']].head(20)

# Display compact result (window_log stored for later use)
final

```
Out[6]:
```
Summary: The filtered window (2021-03-12 12:00–12:30, UTC+8) contains no log records that match the aggregated log series. As a result, window_log is empty and there are no window_count or anomaly_count results (no anomalies detected using the >= p95 rule). If you expected data, please verify the metric timestamps/timezone alignment or expand the time window.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, log_name, p95, window_count, anomaly_count, earliest_anomaly_timestamp, max_value_in_window]
Index: []```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Load error logs
df_errors = pd.read_csv("error_logs.csv")

# Ensure timestamps are localized to UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
df_errors['timestamp'] = pd.to_datetime(df_errors['timestamp'], unit='s').dt.tz_localize(tz)

# Define window (inclusive) in UTC+8
start = pd.Timestamp("2021-03-12 12:00:00", tz=tz)
end = pd.Timestamp("2021-03-12 12:30:00", tz=tz)

# Filter error logs to the time window and store for later inspection
window_error_logs = df_errors[(df_errors['timestamp'] >= start) & (df_errors['timestamp'] <= end)].copy()

# Compact summaries
total_count = window_error_logs.shape[0]
counts_by_cmdb = (
    window_error_logs
    .groupby('cmdb_id', as_index=False)
    .agg(count=('message', 'count'))
    .sort_values('count', ascending=False)
    .head(20)
)
earliest20 = window_error_logs.sort_values('timestamp')[['timestamp', 'cmdb_id', 'message']].head(20)

# Display compact outputs (window_error_logs is stored for later use)
total_count, counts_by_cmdb, earliest20

```
Out[7]:
```
Summary: For the window March 12, 2021 12:00–12:30 (UTC+8) there are no error log rows.

- Total error_log rows in window: 0
- Counts per cmdb_id: none (no rows)
- Earliest raw error_log rows (up to 20): none (no rows)

The filtered-window error logs were stored in the kernel variable window_error_logs but it is empty. If you expected entries, please verify timestamps/timezone alignment or expand the time window.

The original code execution output of IPython Kernel is also provided below for reference:

(0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Helper to ensure timestamp column is datetime tz-aware in UTC+8
def ensure_ts(df, col='timestamp'):
    if col not in df.columns:
        return df
    if not pd.api.types.is_datetime64_any_dtype(df[col]):
        # assume epoch seconds if numeric
        df[col] = pd.to_datetime(df[col], unit='s')
    # If timezone-naive, localize to Asia/Shanghai; if tz-aware, convert to Asia/Shanghai
    if df[col].dt.tz is None:
        df[col] = df[col].dt.tz_localize(tz)
    else:
        df[col] = df[col].dt.tz_convert(tz)
    return df

# Load or reuse metric
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
df_metric = ensure_ts(df_metric, 'timestamp')

# Load or reuse trace
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
df_trace = ensure_ts(df_trace, 'timestamp')

# Load or reuse log
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
df_log = ensure_ts(df_log, 'timestamp')

# Load or reuse error logs
try:
    df_errors
except NameError:
    df_errors = pd.read_csv("error_logs.csv")
df_errors = ensure_ts(df_errors, 'timestamp')

# Function to build summary for a generic dataframe and name column
def build_file_summary(df, file_name, name_col=None, has_value=True):
    total_rows = int(df.shape[0])
    if total_rows == 0:
        min_ts = pd.NaT
        max_ts = pd.NaT
        distinct_ts = 0
    else:
        min_ts = df['timestamp'].min()
        max_ts = df['timestamp'].max()
        distinct_ts = int(df['timestamp'].nunique())
    meta = pd.DataFrame([{
        'file_name': file_name,
        'total_rows': total_rows,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'distinct_timestamp_count': distinct_ts
    }])
    # Prepare earliest and latest rows (limit 10)
    cols = ['timestamp', 'cmdb_id']
    if name_col:
        cols.append(name_col)
    if has_value and 'value' in df.columns:
        cols.append('value')
    # Select only available columns
    cols = [c for c in cols if c in df.columns]
    earliest = df.sort_values('timestamp').loc[:, cols].head(10).reset_index(drop=True)
    latest = df.sort_values('timestamp', ascending=False).loc[:, cols].head(10).reset_index(drop=True)
    return {'meta': meta, 'earliest': earliest, 'latest': latest}

# Build summaries
file_summary_metric = build_file_summary(df_metric, 'metric.csv', name_col='kpi_name', has_value=True)
file_summary_trace = build_file_summary(df_trace, 'trace.csv', name_col='trace_name', has_value=True)
file_summary_log = build_file_summary(df_log, 'log.csv', name_col='log_name', has_value=True)
file_summary_error_logs = build_file_summary(df_errors, 'error_logs.csv', name_col='message', has_value=False)

# Display compact outputs (variables stored for later use)
file_summary_metric, file_summary_trace, file_summary_log, file_summary_error_logs

```
Out[8]:
```
Summary of telemetry file contents (timestamps are UTC+8 / Asia/Shanghai):

- metric.csv
  - Total rows: 27,589
  - Time range: 2021-03-12 04:00:00 → 2021-03-12 04:30:00
  - Distinct timestamps: 31
  - Sample earliest rows (first 3): IG01 JVM-Memory_... (value ~2.04e9), Redis02 connected_clients (402), Redis02 expired_keys (1)
  - Sample latest rows (last 3): dockerB1 container metric (~189523), Mysql01 OSLinux CPU sys time (0.3992), Mysql01 filesystem metrics (various)
  - Stored summary variable: file_summary_metric

- trace.csv
  - Total rows: 7,227
  - Time range: 2021-03-12 04:00:00 → 2021-03-12 04:30:00
  - Distinct timestamps: 31
  - Sample earliest rows (first 3): IG01 trace.from_root.duration_mean (0.0205), Tomcat03 trace.self.row_count (2181), Tomcat03 trace.to_MG01.duration_mean (≈2.799)
  - Sample latest rows (last 3): Tomcat04 trace.self.row_count (1.0), Tomcat04 trace.self.duration_p95 (0.0), MG02 trace.from_Tomcat01.row_count (171)
  - Stored summary variable: file_summary_trace

- log.csv
  - Total rows: 510
  - Time range: 2021-03-12 04:00:00 → 2021-03-12 04:30:00
  - Distinct timestamps: 31
  - Sample earliest rows (first 3): Mysql01 log.error_count (0.0), apache02 log.row_count (688), apache01 log.row_count (684)
  - Sample latest rows (last 3): apache02 log.row_count (15.0), Tomcat03 log.row_count (4.0), Tomcat01 log.error_count (0.0)
  - Stored summary variable: file_summary_log

- error_logs.csv
  - Total rows: 19
  - Time range: 2021-03-12 04:13:26 → 2021-03-12 04:17:01
  - Distinct timestamps: 17
  - Sample earliest rows (first 3): apache02 and apache01 POST request error messages (multiple POST /UOCP/... entries)
  - Sample latest rows (last 3): several apache01 and apache02 POST request messages (timestamps up to 04:17:01)
  - Stored summary variable: file_summary_error_logs

Notes:
- All timestamps in summaries are shown in UTC+8 (Asia/Shanghai).
- Complete per-file summary objects (including up to 10 earliest and 10 latest rows) are available in the kernel variables: file_summary_metric, file_summary_trace, file_summary_log, file_summary_error_logs.

The original code execution output of IPython Kernel is also provided below for reference:

({'meta':     file_name  total_rows             min_timestamp             max_timestamp  distinct_timestamp_count
0  metric.csv       27589 2021-03-12 04:00:00+08:00 2021-03-12 04:30:00+08:00                        31, 'earliest':                   timestamp  cmdb_id                                           kpi_name         value
0 2021-03-12 04:00:00+08:00     IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1 2021-03-12 04:00:00+08:00  Redis02  redis-Redis_6379_Redis  (aof_rewrite_in_progress)  0.000000e+00
2 2021-03-12 04:00:00+08:00  Redis02    redis-Redis_6379_Redis  (aof_rewrite_scheduled)  0.000000e+00
3 2021-03-12 04:00:00+08:00  Redis02          redis-Redis_6379_Redis  (blocked_clients)  0.000000e+00
4 2021-03-12 04:00:00+08:00  Redis02  redis-Redis_6379_Redis  (client_biggest_input_...  0.000000e+00
5 2021-03-12 04:00:00+08:00  Redis02  redis-Redis_6379_Redis  (client_longest_output...  0.000000e+00
6 2021-03-12 04:00:00+08:00  Redis02        redis-Redis_6379_Redis  (connected_clients)  4.020000e+02
7 2021-03-12 04:00:00+08:00  Redis02         redis-Redis_6379_Redis  (connected_slaves)  0.000000e+00
8 2021-03-12 04:00:00+08:00  Redis02             redis-Redis_6379_Redis  (evicted_keys)  0.000000e+00
9 2021-03-12 04:00:00+08:00  Redis02             redis-Redis_6379_Redis  (expired_keys)  1.000000e+00, 'latest':                   timestamp   cmdb_id                                           kpi_name        value
0 2021-03-12 04:30:00+08:00  dockerB1  Container-DOCKER_CONTAINER_464dc801314b--bcou-...  189523.0000
1 2021-03-12 04:30:00+08:00   Mysql01                         OSLinux-CPU_CPU_CPUSysTime       0.3992
2 2021-03-12 04:30:00+08:00   Mysql01  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...       1.1715
3 2021-03-12 04:30:00+08:00   Mysql01        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity      25.2434
4 2021-03-12 04:30:00+08:00   Mysql01  OSLinux-OSLinux_FILESYSTEM_-boot_FSAvailableSpace     341.0000
5 2021-03-12 04:30:00+08:00   Mysql01           OSLinux-OSLinux_FILESYSTEM_-_FSUsedSpace    7019.0000
6 2021-03-12 04:30:00+08:00   Mysql01    OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent      29.6229
7 2021-03-12 04:30:00+08:00   Mysql01            OSLinux-OSLinux_FILESYSTEM_-_FSCapacity      49.1900
8 2021-03-12 04:30:00+08:00   Mysql01      OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace    7251.0000
9 2021-03-12 04:30:00+08:00   Mysql01                        OSLinux-CPU_CPU_CPUidleutil      98.4281}, {'meta':    file_name  total_rows             min_timestamp             max_timestamp  distinct_timestamp_count
0  trace.csv        7227 2021-03-12 04:00:00+08:00 2021-03-12 04:30:00+08:00                        31, 'earliest':                   timestamp   cmdb_id                     trace_name        value
0 2021-03-12 04:00:00+08:00      IG01  trace.from_root.duration_mean     0.020500
1 2021-03-12 04:00:00+08:00  Tomcat03           trace.self.row_count  2181.000000
2 2021-03-12 04:00:00+08:00  Tomcat03    trace.to_MG01.duration_mean     2.799157
3 2021-03-12 04:00:00+08:00  Tomcat03     trace.to_MG01.duration_p95    11.633400
4 2021-03-12 04:00:00+08:00  Tomcat03        trace.to_MG01.row_count   159.000000
5 2021-03-12 04:00:00+08:00  Tomcat03    trace.to_MG02.duration_mean     2.438167
6 2021-03-12 04:00:00+08:00  Tomcat03     trace.to_MG02.duration_p95    10.619900
7 2021-03-12 04:00:00+08:00  Tomcat03        trace.to_MG02.row_count   180.000000
8 2021-03-12 04:00:00+08:00  Tomcat04  trace.from_IG01.duration_mean     2.638451
9 2021-03-12 04:00:00+08:00  Tomcat04   trace.from_IG01.duration_p95    10.016000, 'latest':                   timestamp   cmdb_id                         trace_name       value
0 2021-03-12 04:30:00+08:00  Tomcat04               trace.self.row_count    1.000000
1 2021-03-12 04:30:00+08:00  Tomcat04            trace.self.duration_p95    0.000000
2 2021-03-12 04:30:00+08:00  Tomcat04           trace.self.duration_mean    0.000000
3 2021-03-12 04:29:00+08:00      MG02      trace.from_Tomcat01.row_count  171.000000
4 2021-03-12 04:29:00+08:00      MG02      trace.from_dockerA1.row_count   25.000000
5 2021-03-12 04:29:00+08:00      MG02   trace.from_dockerA1.duration_p95    0.007800
6 2021-03-12 04:29:00+08:00      MG02  trace.from_dockerA1.duration_mean    0.045640
7 2021-03-12 04:29:00+08:00      MG02      trace.from_Tomcat04.row_count  183.000000
8 2021-03-12 04:29:00+08:00      MG02   trace.from_Tomcat04.duration_p95    1.129600
9 2021-03-12 04:29:00+08:00      MG02  trace.from_Tomcat04.duration_mean    0.228705}, {'meta':   file_name  total_rows             min_timestamp             max_timestamp  distinct_timestamp_count
0   log.csv         510 2021-03-12 04:00:00+08:00 2021-03-12 04:30:00+08:00                        31, 'earliest':                   timestamp   cmdb_id          log_name   value
0 2021-03-12 04:00:00+08:00   Mysql01   log.error_count    0.00
1 2021-03-12 04:00:00+08:00  apache02     log.row_count  688.00
2 2021-03-12 04:00:00+08:00  apache02   log.error_count    0.00
3 2021-03-12 04:00:00+08:00  apache01     log.row_count  684.00
4 2021-03-12 04:00:00+08:00  apache01   log.error_count    0.00
5 2021-03-12 04:00:00+08:00  Tomcat04     log.row_count  326.00
6 2021-03-12 04:00:00+08:00  Tomcat04   log.error_count    0.00
7 2021-03-12 04:00:00+08:00  Tomcat03     log.row_count  336.00
8 2021-03-12 04:00:00+08:00  Tomcat03  log.gc_pause_max    0.07
9 2021-03-12 04:00:00+08:00  Tomcat04  log.gc_pause_max    0.10, 'latest':                   timestamp   cmdb_id         log_name  value
0 2021-03-12 04:30:00+08:00  apache02    log.row_count   15.0
1 2021-03-12 04:30:00+08:00  Tomcat03    log.row_count    4.0
2 2021-03-12 04:30:00+08:00  Tomcat01  log.error_count    0.0
3 2021-03-12 04:30:00+08:00  Tomcat01    log.row_count   10.0
4 2021-03-12 04:30:00+08:00  Tomcat02    log.row_count    3.0
5 2021-03-12 04:30:00+08:00  Tomcat03  log.error_count    0.0
6 2021-03-12 04:30:00+08:00  Tomcat02  log.error_count    0.0
7 2021-03-12 04:30:00+08:00  Tomcat04  log.error_count    0.0
8 2021-03-12 04:30:00+08:00  Tomcat04    log.row_count   14.0
9 2021-03-12 04:30:00+08:00  apache01  log.error_count    0.0}, {'meta':         file_name  total_rows             min_timestamp             max_timestamp  distinct_timestamp_count
0  error_logs.csv          19 2021-03-12 04:13:26+08:00 2021-03-12 04:17:01+08:00                        17, 'earliest':                   timestamp   cmdb_id                                            message
0 2021-03-12 04:13:26+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json...
1 2021-03-12 04:13:33+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest3.json...
2 2021-03-12 04:13:41+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json...
3 2021-03-12 04:13:41+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest11.json ...
4 2021-03-12 04:13:50+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H...
5 2021-03-12 04:14:11+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest5.json...
6 2021-03-12 04:14:12+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H...
7 2021-03-12 04:14:19+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest2.json H...
8 2021-03-12 04:14:19+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...
9 2021-03-12 04:14:34+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H..., 'latest':                   timestamp   cmdb_id                                            message
0 2021-03-12 04:17:01+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H...
1 2021-03-12 04:16:45+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H...
2 2021-03-12 04:16:38+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H...
3 2021-03-12 04:16:30+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H...
4 2021-03-12 04:16:29+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...
5 2021-03-12 04:16:07+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
6 2021-03-12 04:15:58+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest4.json H...
7 2021-03-12 04:15:54+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest1.json...
8 2021-03-12 04:15:36+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest2.json H...
9 2021-03-12 04:14:34+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H...})```
```

In [ ]:
# In[9]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing df_metric and agg_metric if present; otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timezone variable and timestamp tz-awareness (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if not pd.api.types.is_datetime64_any_dtype(df_metric['timestamp']):
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s')
# Localize or convert to Asia/Shanghai
if df_metric['timestamp'].dt.tz is None:
    df_metric['timestamp'] = df_metric['timestamp'].dt.tz_localize(tz)
else:
    df_metric['timestamp'] = df_metric['timestamp'].dt.tz_convert(tz)

# Ensure agg_metric (global p95 per series) exists; compute if missing
try:
    agg_metric
except NameError:
    agg_metric = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(p95=('value', lambda s: s.quantile(0.95) if s.notna().any() else np.nan))
    )

# Define telemetry-covered incident window in UTC+8 (inclusive)
start = pd.Timestamp("2021-03-12 04:00:00", tz=tz)
end = pd.Timestamp("2021-03-12 04:30:00", tz=tz)

# Filter metric records to the window and store for later use
window_metric = df_metric[(df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end)].copy()

# Merge windowed records with global p95 thresholds (only consider known series)
merged = window_metric.merge(
    agg_metric[['cmdb_id', 'kpi_name', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='inner'
)

# If no records, prepare empty final table
if merged.empty:
    final = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'p95', 'window_count', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window'])
else:
    # Mark anomalies using global p95 (value >= p95). Treat NaN p95 as non-anomalous.
    merged['is_anomaly'] = (~merged['p95'].isna()) & (merged['value'] >= merged['p95'])

    # Aggregate overall window stats
    grp_all = merged.groupby(['cmdb_id', 'kpi_name', 'p95'], as_index=False).agg(
        window_count=('value', 'count'),
        max_value_in_window=('value', 'max')
    )

    # Aggregate anomaly-specific stats
    grp_anom = (
        merged[merged['is_anomaly']]
        .groupby(['cmdb_id', 'kpi_name', 'p95'], as_index=False)
        .agg(
            anomaly_count=('is_anomaly', 'sum'),
            earliest_anomaly_timestamp=('timestamp', 'min')
        )
    )

    # Combine aggregates
    summary = grp_all.merge(grp_anom, on=['cmdb_id', 'kpi_name', 'p95'], how='left')
    summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)

    # Compute severity ratio for sorting (handle p95 == 0 or NaN)
    with np.errstate(divide='ignore', invalid='ignore'):
        summary['severity_ratio'] = np.where(
            summary['p95'] > 0,
            summary['max_value_in_window'] / summary['p95'],
            np.where(summary['max_value_in_window'] > 0, np.inf, 0)
        )

    # Sort by anomaly_count desc, then severity_ratio desc
    summary = summary.sort_values(by=['anomaly_count', 'severity_ratio'], ascending=[False, False])

    # Prepare compact final view (requested columns) limited to top 20
    final = summary[['cmdb_id', 'kpi_name', 'p95', 'window_count', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window']].head(20)

# Display the compact result; window_metric is stored for later use
final

```
Out[9]:
```
summary = (
    "Summary for metric window 2021-03-12 04:00:00 to 04:30:00 (UTC+8) using rule value >= global P95:\n"
    "- Many component-KPI series had every point in the 31-minute window flagged as anomalous (window_count = anomaly_count = 31).\n"
    "- Notable impacted series (examples from the top results):\n"
    "  * Redis02.redis (keyspace_misses): p95=23072, max=23080, anomalies at 2021-03-12 04:00:00+08:00\n"
    "  * IG01 JVM-Memory_HeapMemoryMax and NoHeapMemoryUsed: p95≈2.040136e9 and ≈5.789e7, max values equal those p95s, anomalies from 04:00:00+08:00\n"
    "  * IG02, MG01, MG02 have similar JVM memory KPIs (HeapMemoryMax and NoHeapMemoryUsed) with all-window anomalies.\n"
    "  * Several OS/process KPIs (e.g., PROCESS_PROCNoZombies, zabbix_PROCPPCount) across IG, MG, Mysql, Redis also show anomalies for all 31 points.\n"
    "  * Redis01.connected_clients: p95=602, window max=602, anomalies across the window.\n"
    "- Earliest anomaly timestamps for the reported series are at the start of the window (2021-03-12 04:00:00+08:00), meaning elevated values were present from the first sampled minute.\n"
    "- Interpretation: the global P95 thresholds for these series are at or below the observed values in this incident window, so the >=P95 rule marks the entire window as anomalous for these KPIs. "
    "You may want to review how global P95 was computed or consider alternative anomaly thresholds or per-window baselines if this produces many full-window matches."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name           p95  window_count  anomaly_count earliest_anomaly_timestamp  max_value_in_window
1011  Redis02          redis-Redis_6379_Redis  (keyspace_misses)  2.307200e+04            31             31  2021-03-12 04:00:00+08:00         2.308000e+04
0        IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09            31             31  2021-03-12 04:00:00+08:00         2.040136e+09
3        IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.789193e+07            31             31  2021-03-12 04:00:00+08:00         5.789193e+07
77       IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00            31             31  2021-03-12 04:00:00+08:00         1.000000e+00
80       IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01            31             31  2021-03-12 04:00:00+08:00         3.000000e-01
81       IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00            31             31  2021-03-12 04:00:00+08:00         3.000000e+00
88       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09            31             31  2021-03-12 04:00:00+08:00         2.040136e+09
91       IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.788924e+07            31             31  2021-03-12 04:00:00+08:00         5.788924e+07
165      IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00            31             31  2021-03-12 04:00:00+08:00         1.000000e+00
168      IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01            31             31  2021-03-12 04:00:00+08:00         3.000000e-01
169      IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00            31             31  2021-03-12 04:00:00+08:00         3.000000e+00
176      MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09            31             31  2021-03-12 04:00:00+08:00         2.040136e+09
179      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.069495e+08            31             31  2021-03-12 04:00:00+08:00         5.069495e+08
268      MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09            31             31  2021-03-12 04:00:00+08:00         2.040136e+09
271      MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  9.471521e+08            31             31  2021-03-12 04:00:00+08:00         9.471521e+08
578   Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  7.000000e+00            31             31  2021-03-12 04:00:00+08:00         7.000000e+00
802   Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  2.400000e+01            31             31  2021-03-12 04:00:00+08:00         2.400000e+01
879   Redis01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00            31             31  2021-03-12 04:00:00+08:00         1.000000e+00
883   Redis01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  4.000000e+00            31             31  2021-03-12 04:00:00+08:00         4.000000e+00
894   Redis01        redis-Redis_6379_Redis  (connected_clients)  6.020000e+02            31             31  2021-03-12 04:00:00+08:00         6.020000e+02```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_log if present, otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure timestamp column is datetime and localized to UTC+8
if not pd.api.types.is_datetime64_any_dtype(df_log['timestamp']):
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s')
if df_log['timestamp'].dt.tz is None:
    df_log['timestamp'] = df_log['timestamp'].dt.tz_localize(tz)
else:
    df_log['timestamp'] = df_log['timestamp'].dt.tz_convert(tz)

# Define window (inclusive) in UTC+8
start = pd.Timestamp("2021-03-12 04:00:00", tz=tz)
end = pd.Timestamp("2021-03-12 04:30:00", tz=tz)

# Filter log records to the window and store for later use
window_log = df_log[(df_log['timestamp'] >= start) & (df_log['timestamp'] <= end)].copy()

# Prepare per-cmdb_id summary
# total_rows_in_window
grp_total = window_log.groupby('cmdb_id', as_index=False).agg(total_rows_in_window=('timestamp', 'count'))

# error_count_rows and earliest_error_timestamp where log_name == 'log.error_count' and value > 0
mask_err = (window_log['log_name'] == 'log.error_count') & (window_log['value'] > 0)
err_rows = window_log[mask_err]

grp_err = err_rows.groupby('cmdb_id', as_index=False).agg(
    error_count_rows=('timestamp', 'count'),
    earliest_error_timestamp=('timestamp', 'min')
)

# Combine totals and error stats (ensure cmdb_id present even if zero errors)
summary_table = grp_total.merge(grp_err, on='cmdb_id', how='left')
summary_table['error_count_rows'] = summary_table['error_count_rows'].fillna(0).astype(int)
# earliest_error_timestamp remains NaT where no errors

# Sort and limit to top 20 by error_count_rows desc
summary_table = summary_table.sort_values(by='error_count_rows', ascending=False).head(20).reset_index(drop=True)

# Prepare up to earliest 20 raw log rows in window
earliest20_logs = window_log.sort_values('timestamp').loc[:, ['timestamp', 'cmdb_id', 'log_name', 'value']].head(20).reset_index(drop=True)

# Display compact outputs and keep window_log for later use
summary_table, earliest20_logs

```
Out[10]:
```
Summary for log data in window 2021-03-12 04:00:00 → 04:30:00 (UTC+8):

- Per-cmdb_id summary (sorted by error_count_rows):
  - apache01: total_rows_in_window = 62, error_count_rows = 5, earliest_error_timestamp = 2021-03-12 04:13:00+08:00
  - apache02: total_rows_in_window = 62, error_count_rows = 4, earliest_error_timestamp = 2021-03-12 04:13:00+08:00
  - Mysql01: total_rows_in_window = 52, error_count_rows = 0
  - Tomcat02: total_rows_in_window = 92, error_count_rows = 0
  - Tomcat01: total_rows_in_window = 81, error_count_rows = 0
  - Tomcat04: total_rows_in_window = 80, error_count_rows = 0
  - Tomcat03: total_rows_in_window = 81, error_count_rows = 0

- Earliest raw log rows in the window (first ~20): show initial per-minute log.row_count, log.error_count, and log.gc_pause_max values at and after 04:00:00 (examples include apache and Tomcat row_count entries around 04:00 and subsequent per-minute values). The first nonzero error_count entries for apache01/apache02 appear around 04:13:00.

- The filtered-window log rows were stored in the kernel variable window_log for later use.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  total_rows_in_window  error_count_rows  earliest_error_timestamp
0  apache01                    62                 5 2021-03-12 04:13:00+08:00
1  apache02                    62                 4 2021-03-12 04:13:00+08:00
2   Mysql01                    52                 0                       NaT
3  Tomcat02                    92                 0                       NaT
4  Tomcat01                    81                 0                       NaT
5  Tomcat04                    80                 0                       NaT
6  Tomcat03                    81                 0                       NaT,                    timestamp   cmdb_id          log_name   value
0  2021-03-12 04:00:00+08:00   Mysql01   log.error_count    0.00
1  2021-03-12 04:00:00+08:00  apache02     log.row_count  688.00
2  2021-03-12 04:00:00+08:00  apache02   log.error_count    0.00
3  2021-03-12 04:00:00+08:00  apache01     log.row_count  684.00
4  2021-03-12 04:00:00+08:00  apache01   log.error_count    0.00
5  2021-03-12 04:00:00+08:00  Tomcat04     log.row_count  326.00
6  2021-03-12 04:00:00+08:00  Tomcat04   log.error_count    0.00
7  2021-03-12 04:00:00+08:00  Tomcat03     log.row_count  336.00
8  2021-03-12 04:00:00+08:00  Tomcat03  log.gc_pause_max    0.07
9  2021-03-12 04:00:00+08:00  Tomcat04  log.gc_pause_max    0.10
10 2021-03-12 04:00:00+08:00  Tomcat02     log.row_count  352.00
11 2021-03-12 04:00:00+08:00  Tomcat02  log.gc_pause_max    0.08
12 2021-03-12 04:00:00+08:00  Tomcat02   log.error_count    0.00
13 2021-03-12 04:00:00+08:00  Tomcat01     log.row_count  336.00
14 2021-03-12 04:00:00+08:00  Tomcat01  log.gc_pause_max    0.07
15 2021-03-12 04:00:00+08:00  Tomcat01   log.error_count    0.00
16 2021-03-12 04:00:00+08:00   Mysql01     log.row_count   53.00
17 2021-03-12 04:00:00+08:00  Tomcat03   log.error_count    0.00
18 2021-03-12 04:01:00+08:00  Tomcat03     log.row_count  392.00
19 2021-03-12 04:01:00+08:00  apache02     log.row_count  749.00)```
```

In [ ]:
# In[11]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing df_trace and agg_trace if present
tz = pytz.timezone('Asia/Shanghai')
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s').dt.tz_localize(tz)

# Ensure timestamp is tz-aware in Asia/Shanghai
if not pd.api.types.is_datetime64_any_dtype(df_trace['timestamp']):
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s')
if df_trace['timestamp'].dt.tz is None:
    df_trace['timestamp'] = df_trace['timestamp'].dt.tz_localize(tz)
else:
    df_trace['timestamp'] = df_trace['timestamp'].dt.tz_convert(tz)

# Ensure agg_trace (global p95 per series) exists
try:
    agg_trace
except NameError:
    agg_trace = (
        df_trace.groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(p95=('value', lambda s: s.quantile(0.95) if s.notna().any() else np.nan))
    )

# Define telemetry-covered incident window (UTC+8)
start = pd.Timestamp("2021-03-12 04:00:00", tz=tz)
end = pd.Timestamp("2021-03-12 04:30:00", tz=tz)

# Filter trace records to the window and store for later use
window_trace = df_trace[(df_trace['timestamp'] >= start) & (df_trace['timestamp'] <= end)].copy()

# Merge windowed records with precomputed global p95 thresholds
merged = window_trace.merge(
    agg_trace[['cmdb_id', 'trace_name', 'p95']],
    on=['cmdb_id', 'trace_name'],
    how='inner'
)

# If no records, prepare empty final table
if merged.empty:
    final = pd.DataFrame(columns=[
        'cmdb_id', 'trace_name', 'p95', 'window_count', 'anomaly_count',
        'earliest_anomaly_timestamp', 'max_value_in_window'
    ])
else:
    # Mark anomalies using global p95 (value >= p95). Treat NaN p95 as non-anomalous.
    merged['is_anomaly'] = (~merged['p95'].isna()) & (merged['value'] >= merged['p95'])

    # Aggregate window-level stats
    grp_all = merged.groupby(['cmdb_id', 'trace_name', 'p95'], as_index=False).agg(
        window_count=('value', 'count'),
        max_value_in_window=('value', 'max')
    )

    # Aggregate anomaly-specific stats
    grp_anom = (
        merged[merged['is_anomaly']]
        .groupby(['cmdb_id', 'trace_name', 'p95'], as_index=False)
        .agg(
            anomaly_count=('is_anomaly', 'sum'),
            earliest_anomaly_timestamp=('timestamp', 'min')
        )
    )

    # Combine aggregates, fill missing anomaly info
    summary = grp_all.merge(grp_anom, on=['cmdb_id', 'trace_name', 'p95'], how='left')
    summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)

    # Compute severity ratio for sorting: max_value_in_window / p95 (handle p95 == 0 or NaN)
    with np.errstate(divide='ignore', invalid='ignore'):
        summary['severity_ratio'] = np.where(
            summary['p95'] > 0,
            summary['max_value_in_window'] / summary['p95'],
            np.where(summary['max_value_in_window'] > 0, np.inf, 0)
        )

    # Sort by anomaly_count desc, then severity_ratio desc
    summary = summary.sort_values(by=['anomaly_count', 'severity_ratio'], ascending=[False, False])

    # Prepare compact final table and limit to top 20
    final = summary[[
        'cmdb_id', 'trace_name', 'p95', 'window_count', 'anomaly_count',
        'earliest_anomaly_timestamp', 'max_value_in_window'
    ]].head(20)

# Display compact result (window_trace stored for follow-up)
final

```
Out[11]:
```
Summary of trace anomalies in window 2021-03-12 04:00–04:30 (UTC+8), using rule value >= global P95:

- Overall: Several trace series exceeded their global P95 during the 30-minute window. Most series have 30 samples in the window (Tomcat04 has 31).

- Top findings (examples from the sorted results):
  - dockerA1.trace.self.duration_p95 — p95=1.008, window_count=30, anomaly_count=4, earliest anomaly 2021-03-12 04:00:00, max=1.009 (small exceedances).
  - MG01.trace.to_dockerA1.row_count and dockerA1.trace.from_MG01.row_count — p95=164, window_count=30, anomaly_count=3, earliest anomaly 04:03:00, max=185 (substantial row_count spikes).
  - MG02.trace.from_Tomcat04.row_count and Tomcat04.trace.to_MG02.row_count — p95=190, window_count=30, anomaly_count=3, earliest anomaly 04:01:00, max=192.
  - Several duration-mean and duration-p95 series show repeated spikes (anomaly_count=2), for example:
    - MG01.trace.from_dockerA1.duration_mean & dockerA1.trace.to_MG01.duration_mean — p95≈6.27, max≈12.06, earliest anomaly 04:01:00 (≈2x the p95).
    - MG02.trace.to_dockerB1.duration_mean & dockerB1.trace.from_MG02.duration_mean — p95≈6.55, max≈10.92, earliest anomaly 04:02:00.
    - Pairs of trace.duration_p95 for Tomcat↔MG flows show p95s in the ~13–22 range with max values up to ~32 (notable latency spikes at ~04:02–04:05).

- Timing: Earliest anomalies appear as early as 04:00:00, with many series first exceeding P95 between 04:00 and 04:05; a later cluster (e.g., some row_count anomalies) occurs around 04:16 for certain series.

- Affected components (not exhaustive): dockerA1, MG01, MG02, Tomcat01/02/03/04, dockerB1 (and related inter-service traces).

- Interpretation / next steps: anomalies are concentrated on inter-service row_count and trace duration metrics (some >1.5–2x p95). Prioritize investigating dockerA1 and MG01/MG02 interactions and the Tomcat→MG flows (latency and elevated row counts) around 04:00–04:05.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name         p95  window_count  anomaly_count earliest_anomaly_timestamp  max_value_in_window
202  dockerA1            trace.self.duration_p95    1.008000            30              4  2021-03-12 04:00:00+08:00             1.009000
68       MG01        trace.to_dockerA1.row_count  164.000000            30              3  2021-03-12 04:03:00+08:00           185.000000
194  dockerA1          trace.from_MG01.row_count  164.000000            30              3  2021-03-12 04:03:00+08:00           185.000000
89       MG02      trace.from_Tomcat04.row_count  190.000000            30              3  2021-03-12 04:01:00+08:00           192.000000
191  Tomcat04            trace.to_MG02.row_count  190.000000            30              3  2021-03-12 04:01:00+08:00           192.000000
48       MG01  trace.from_dockerA1.duration_mean    6.271071            30              2  2021-03-12 04:01:00+08:00            12.063615
204  dockerA1        trace.to_MG01.duration_mean    6.271071            30              2  2021-03-12 04:01:00+08:00            12.063615
114      MG02    trace.to_dockerB1.duration_mean    6.552625            30              2  2021-03-12 04:02:00+08:00            10.924569
231  dockerB1      trace.from_MG02.duration_mean    6.552625            30              2  2021-03-12 04:02:00+08:00            10.924569
49       MG01   trace.from_dockerA1.duration_p95   13.190138            30              2  2021-03-12 04:02:00+08:00            21.146800
205  dockerA1         trace.to_MG01.duration_p95   13.190138            30              2  2021-03-12 04:02:00+08:00            21.146800
184  Tomcat04            trace.self.duration_p95    3.609700            31              2  2021-03-12 04:03:00+08:00             5.684900
37       MG01   trace.from_Tomcat01.duration_p95   18.549597            30              2  2021-03-12 04:05:00+08:00            27.769000
133  Tomcat01         trace.to_MG01.duration_p95   18.549597            30              2  2021-03-12 04:05:00+08:00            27.769000
85       MG02   trace.from_Tomcat03.duration_p95   21.688257            30              2  2021-03-12 04:05:00+08:00            32.187000
172  Tomcat03         trace.to_MG02.duration_p95   21.688257            30              2  2021-03-12 04:05:00+08:00            32.187000
106      MG02            trace.self.duration_p95   16.541160            30              2  2021-03-12 04:02:00+08:00            24.066050
82       MG02   trace.from_Tomcat02.duration_p95   22.244230            30              2  2021-03-12 04:02:00+08:00            31.926000
154  Tomcat02         trace.to_MG02.duration_p95   22.244230            30              2  2021-03-12 04:02:00+08:00            31.926000
101      MG02      trace.from_dockerB2.row_count  228.950000            30              2  2021-03-12 04:16:00+08:00           322.000000```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Desired time window for earliest_anomaly_timestamp (inclusive), UTC+8
start = pd.Timestamp("2021-03-12 04:12:00", tz=tz)
end = pd.Timestamp("2021-03-12 04:17:00", tz=tz)

# Attempt to reuse an existing aggregated anomaly table if available (e.g., 'final')
# Otherwise, compute the aggregated windowed anomaly summary from window_trace + agg_trace.
needed_cols = {'cmdb_id', 'trace_name', 'p95', 'window_count', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window'}
try:
    candidate = final  # 'final' was the name used previously for the aggregated trace summary
    if not needed_cols.issubset(set(candidate.columns)):
        raise NameError("final exists but lacks required columns")
except Exception:
    # Build aggregated summary from window_trace and agg_trace
    # Ensure window_trace and agg_trace exist
    if 'window_trace' not in globals():
        raise NameError("window_trace not found in kernel. Please run the windowed trace aggregation step first.")
    if 'agg_trace' not in globals():
        raise NameError("agg_trace not found in kernel. Please run the global trace aggregation step first.")
    # Merge and compute anomalies using global p95 thresholds
    merged = window_trace.merge(
        agg_trace[['cmdb_id', 'trace_name', 'p95']],
        on=['cmdb_id', 'trace_name'],
        how='inner'
    ).copy()
    merged['is_anomaly'] = (~merged['p95'].isna()) & (merged['value'] >= merged['p95'])
    grp_all = merged.groupby(['cmdb_id', 'trace_name', 'p95'], as_index=False).agg(
        window_count=('value', 'count'),
        max_value_in_window=('value', 'max')
    )
    grp_anom = (
        merged[merged['is_anomaly']]
        .groupby(['cmdb_id', 'trace_name', 'p95'], as_index=False)
        .agg(
            anomaly_count=('is_anomaly', 'sum'),
            earliest_anomaly_timestamp=('timestamp', 'min')
        )
    )
    candidate = grp_all.merge(grp_anom, on=['cmdb_id', 'trace_name', 'p95'], how='left')
    candidate['anomaly_count'] = candidate['anomaly_count'].fillna(0).astype(int)

# Ensure earliest_anomaly_timestamp is tz-aware in Asia/Shanghai
if 'earliest_anomaly_timestamp' in candidate.columns and not pd.api.types.is_datetime64_any_dtype(candidate['earliest_anomaly_timestamp']):
    candidate['earliest_anomaly_timestamp'] = pd.to_datetime(candidate['earliest_anomaly_timestamp'])
if 'earliest_anomaly_timestamp' in candidate.columns and candidate['earliest_anomaly_timestamp'].dt.tz is None:
    candidate['earliest_anomaly_timestamp'] = candidate['earliest_anomaly_timestamp'].dt.tz_localize(tz)

# Filter rows where earliest_anomaly_timestamp is within [start, end]
mask = candidate['earliest_anomaly_timestamp'].notna() & \
       (candidate['earliest_anomaly_timestamp'] >= start) & \
       (candidate['earliest_anomaly_timestamp'] <= end)

trace_anomalies_apache_window = candidate.loc[mask, [
    'cmdb_id', 'trace_name', 'p95', 'window_count', 'anomaly_count',
    'earliest_anomaly_timestamp', 'max_value_in_window'
]].sort_values(by=['earliest_anomaly_timestamp', 'anomaly_count'], ascending=[True, False]).head(20)

# Display the compact result (and store trace_anomalies_apache_window for later use)
trace_anomalies_apache_window

```
Out[12]:
```
summary = (
    "Summary: Between 2021-03-12 04:12:00 and 04:17:00 (UTC+8) there is a single trace series with an earliest anomaly in that interval: "
    "MG02.trace.from_dockerB2.row_count (global p95 = 228.95). In the 04:00–04:30 window this series had 30 points, "
    "2 of which met or exceeded the p95 (anomalies). The earliest anomaly occurred at 2021-03-12 04:16:00+08:00 and the maximum value in the window was 322.0."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                     trace_name     p95  window_count  anomaly_count earliest_anomaly_timestamp  max_value_in_window
101    MG02  trace.from_dockerB2.row_count  228.95            30              2  2021-03-12 04:16:00+08:00                322.0```
```

In [ ]:
# In[13]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_metric and agg_metric if present; otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timestamp is datetime tz-aware in Asia/Shanghai
if not pd.api.types.is_datetime64_any_dtype(df_metric['timestamp']):
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s')
if df_metric['timestamp'].dt.tz is None:
    df_metric['timestamp'] = df_metric['timestamp'].dt.tz_localize(tz)
else:
    df_metric['timestamp'] = df_metric['timestamp'].dt.tz_convert(tz)

try:
    agg_metric
except NameError:
    # Compute global p95 per component-KPI if missing (must be global before filtering)
    agg_metric = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(p95=('value', lambda s: s.quantile(0.95) if s.notna().any() else np.nan))
    )

# Define telemetry-covered window (UTC+8)
start = pd.Timestamp("2021-03-12 04:00:00", tz=tz)
end = pd.Timestamp("2021-03-12 04:30:00", tz=tz)

# Filter metric rows to the window and Apache components; store for later use
apache_ids = ['apache01', 'apache02']
apache_window_metric = df_metric[
    (df_metric['timestamp'] >= start) &
    (df_metric['timestamp'] <= end) &
    (df_metric['cmdb_id'].isin(apache_ids))
].copy()

# Merge with global p95 thresholds (only consider series present in agg_metric)
merged = apache_window_metric.merge(
    agg_metric[['cmdb_id', 'kpi_name', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='inner'
)

if merged.empty:
    apache_summary = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'p95', 'window_count', 'anomaly_count',
        'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio'
    ])
else:
    # Mark anomalies using global p95 (value >= p95). NaN p95 => not anomaly.
    merged['is_anomaly'] = (~merged['p95'].isna()) & (merged['value'] >= merged['p95'])

    grp_all = merged.groupby(['cmdb_id', 'kpi_name', 'p95'], as_index=False).agg(
        window_count=('value', 'count'),
        max_value_in_window=('value', 'max')
    )

    grp_anom = (
        merged[merged['is_anomaly']]
        .groupby(['cmdb_id', 'kpi_name', 'p95'], as_index=False)
        .agg(
            anomaly_count=('is_anomaly', 'sum'),
            earliest_anomaly_timestamp=('timestamp', 'min')
        )
    )

    apache_summary = grp_all.merge(grp_anom, on=['cmdb_id', 'kpi_name', 'p95'], how='left')
    apache_summary['anomaly_count'] = apache_summary['anomaly_count'].fillna(0).astype(int)

    # severity_ratio: max_value_in_window / p95 if p95>0 else NaN
    apache_summary['severity_ratio'] = np.where(
        apache_summary['p95'] > 0,
        apache_summary['max_value_in_window'] / apache_summary['p95'],
        np.nan
    )

    # Sort by anomaly_count desc then severity_ratio desc
    apache_summary = apache_summary.sort_values(by=['anomaly_count', 'severity_ratio'], ascending=[False, False])

# Limit to top 20 and select requested columns
apache_summary = apache_summary[[
    'cmdb_id', 'kpi_name', 'p95', 'window_count', 'anomaly_count',
    'earliest_anomaly_timestamp', 'max_value_in_window', 'severity_ratio'
]].head(20)

# Display result; apache_window_metric is stored for later use
apache_summary

```
Out[13]:
```
Summary of Apache metrics in window 2021-03-12 04:00–04:30 (UTC+8):

- Many apache01 and apache02 metric series were flagged as anomalous for essentially the entire window (window_count ≈ anomaly_count, often 31 or 30). Earliest anomalies mostly occur at the start of the window (04:00:00 or 04:01:00).
- Examples of fully-window anomalous series:
  - apache01: apache_10001_PROCPPCount (p95=10), zabbix-related process counts, PROCESS_PROCNoZombies, many local-disk and network KPIs — anomaly_count = window_count.
  - apache02: similar zabbix/process KPIs and apache_10001_PROCPPCount (p95=10) — anomaly_count = window_count.
- Memory-related spikes:
  - apache01 OSLinux MEMORY_MEMUsedMemPerc: p95=67, window_count=21, anomaly_count=21 (severity_ratio = 1.0).
  - apache02 OSLinux MEMORY_MEMUsedMemPerc: p95=74, window_count=20, anomaly_count=20 (severity_ratio = 1.0).
- Many metrics have p95 == observed max (severity_ratio ≈ 1.0). Several network/disk KPIs have p95 == 0 so severity_ratio is undefined (NaN).
- Interpretation: the global P95 thresholds for these Apache series are at or below the observed values in this incident window, causing the >=p95 rule to mark most Apache metrics as anomalous across the whole window. Recommend reviewing how global P95 was computed or using alternative/local baselines or thresholds to reduce full-window matches before triage.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    p95  window_count  anomaly_count earliest_anomaly_timestamp  max_value_in_window  severity_ratio
71   apache01   OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCount   10.0            31             31  2021-03-12 04:00:00+08:00                 10.0             1.0
72   apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    0.5            31             31  2021-03-12 04:00:00+08:00                  0.5             1.0
73   apache01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    4.0            31             31  2021-03-12 04:00:00+08:00                  4.0             1.0
152  apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    0.4            31             31  2021-03-12 04:00:00+08:00                  0.4             1.0
153  apache02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    7.0            31             31  2021-03-12 04:00:00+08:00                  7.0             1.0
66   apache01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    0.0            31             31  2021-03-12 04:00:00+08:00                  0.0             NaN
68   apache01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    0.0            31             31  2021-03-12 04:00:00+08:00                  0.0             NaN
146  apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    0.0            31             31  2021-03-12 04:00:00+08:00                  0.0             NaN
148  apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    0.0            31             31  2021-03-12 04:00:00+08:00                  0.0             NaN
151  apache02   OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCount   10.0            30             30  2021-03-12 04:00:00+08:00                 10.0             1.0
145  apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.0            30             30  2021-03-12 04:00:00+08:00                  0.0             NaN
62   apache01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc   67.0            21             21  2021-03-12 04:00:00+08:00                 67.0             1.0
69   apache01          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem  124.0            20             20  2021-03-12 04:01:00+08:00                124.0             1.0
142  apache02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc   74.0            20             20  2021-03-12 04:01:00+08:00                 74.0             1.0
43   apache01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.0            15             15  2021-03-12 04:00:00+08:00                  0.0             NaN
45   apache01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    0.0            15             15  2021-03-12 04:00:00+08:00                  0.0             NaN
46   apache01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    0.0            15             15  2021-03-12 04:00:00+08:00                  0.0             NaN
54   apache01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps    0.0            15             15  2021-03-12 04:00:00+08:00                  0.0             NaN
55   apache01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.0            15             15  2021-03-12 04:00:00+08:00                  0.0             NaN
70   apache01      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMemPerc    0.0            15             15  2021-03-12 04:01:00+08:00                  0.0             NaN```
```

In [ ]:
# In[14]:

import pandas as pd
import pytz

# Reuse df_errors if present; otherwise load
try:
    df_errors
except NameError:
    df_errors = pd.read_csv("error_logs.csv")

# Ensure timestamp is datetime tz-aware in Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
if not pd.api.types.is_datetime64_any_dtype(df_errors['timestamp']):
    df_errors['timestamp'] = pd.to_datetime(df_errors['timestamp'], unit='s')
if df_errors['timestamp'].dt.tz is None:
    df_errors['timestamp'] = df_errors['timestamp'].dt.tz_localize(tz)
else:
    df_errors['timestamp'] = df_errors['timestamp'].dt.tz_convert(tz)

# Define interval (inclusive) in UTC+8
start = pd.Timestamp("2021-03-12 04:12:00", tz=tz)
end = pd.Timestamp("2021-03-12 04:17:30", tz=tz)

# Filter and store for later inspection
apache_error_messages_window = df_errors[(df_errors['timestamp'] >= start) & (df_errors['timestamp'] <= end)].copy()

# Compact summaries
total_count = int(apache_error_messages_window.shape[0])

counts_by_cmdb = (
    apache_error_messages_window
    .groupby('cmdb_id', as_index=False)
    .agg(count=('message', 'count'))
    .sort_values('count', ascending=False)
    .head(20)
)

# Prepare truncated message prefix (<=200 chars) for grouping/display
apache_error_messages_window['message_short'] = apache_error_messages_window['message'].astype(str).str.slice(0,200)

top_messages = (
    apache_error_messages_window
    .groupby('message_short', as_index=False)
    .agg(
        count=('message', 'count'),
        earliest_occurrence=('timestamp', 'min')
    )
    .sort_values(['count', 'earliest_occurrence'], ascending=[False, True])
    .head(20)
)

# Earliest and latest raw error log rows in the interval (compact: timestamp, cmdb_id, message)
if not apache_error_messages_window.empty:
    earliest_row = apache_error_messages_window.sort_values('timestamp').loc[:, ['timestamp', 'cmdb_id', 'message']].head(1).reset_index(drop=True)
    latest_row = apache_error_messages_window.sort_values('timestamp', ascending=False).loc[:, ['timestamp', 'cmdb_id', 'message']].head(1).reset_index(drop=True)
else:
    earliest_row = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'message'])
    latest_row = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'message'])

# Display compact outputs (variables stored for later use)
total_count, counts_by_cmdb, top_messages, earliest_row, latest_row

```
Out[14]:
```
summary = (
    "Summary of error logs between 2021-03-12 04:12:00 and 04:17:30 (UTC+8):\n"
    "- Total rows: 19\n"
    "- By host: apache01 = 10 rows, apache02 = 9 rows\n"
    "- Messages: 19 distinct error messages (each occurred once). They are HTTP POST-related entries targeting various /UOCP/... ServiceTest endpoints (e.g., ServiceTest1/2/3/4/5/6/7/9/10/11). Message texts are short POST request traces and were grouped by their first ~200 chars in the analysis.\n"
    "- Earliest row in interval: 2021-03-12 04:13:26+08:00 (apache02)\n"
    "- Latest row in interval: 2021-03-12 04:17:01+08:00 (apache01)\n"
    "- The filtered window rows are stored in kernel variable: apache_error_messages_window\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(19,     cmdb_id  count
0  apache01     10
1  apache02      9,                                         message_short  count       earliest_occurrence
16  IPAddress "POST /UOCP/person/ServiceTest5.json...      1 2021-03-12 04:13:26+08:00
15  IPAddress "POST /UOCP/person/ServiceTest3.json...      1 2021-03-12 04:13:33+08:00
0   IPAddress "POST /UOCP/base/ServiceTest11.json ...      1 2021-03-12 04:13:41+08:00
11  IPAddress "POST /UOCP/person/ServiceTest1.json...      1 2021-03-12 04:13:41+08:00
6   IPAddress "POST /UOCP/base/ServiceTest6.json H...      1 2021-03-12 04:13:50+08:00
17  IPAddress "POST /UOCP/person/ServiceTest5.json...      1 2021-03-12 04:14:11+08:00
10  IPAddress "POST /UOCP/base/ServiceTest7.json H...      1 2021-03-12 04:14:12+08:00
2   IPAddress "POST /UOCP/base/ServiceTest2.json H...      1 2021-03-12 04:14:19+08:00
14  IPAddress "POST /UOCP/person/ServiceTest10.jso...      1 2021-03-12 04:14:19+08:00
8   IPAddress "POST /UOCP/base/ServiceTest7.json H...      1 2021-03-12 04:14:34+08:00
1   IPAddress "POST /UOCP/base/ServiceTest2.json H...      1 2021-03-12 04:15:36+08:00
12  IPAddress "POST /UOCP/person/ServiceTest1.json...      1 2021-03-12 04:15:54+08:00
3   IPAddress "POST /UOCP/base/ServiceTest4.json H...      1 2021-03-12 04:15:58+08:00
18  IPAddress "POST /UOCP/person/ServiceTest9.json...      1 2021-03-12 04:16:07+08:00
13  IPAddress "POST /UOCP/person/ServiceTest10.jso...      1 2021-03-12 04:16:29+08:00
4   IPAddress "POST /UOCP/base/ServiceTest4.json H...      1 2021-03-12 04:16:30+08:00
7   IPAddress "POST /UOCP/base/ServiceTest7.json H...      1 2021-03-12 04:16:38+08:00
5   IPAddress "POST /UOCP/base/ServiceTest4.json H...      1 2021-03-12 04:16:45+08:00
9   IPAddress "POST /UOCP/base/ServiceTest7.json H...      1 2021-03-12 04:17:01+08:00,                   timestamp   cmdb_id                                            message
0 2021-03-12 04:13:26+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json...,                   timestamp   cmdb_id                                            message
0 2021-03-12 04:17:01+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H...)```
```

In [ ]:
# In[15]:

# Display up to 50 raw error log rows previously filtered to 2021-03-12 04:12:00 → 04:17:30 (UTC+8)
# Reuse the in-kernel variable 'apache_error_messages_window' and show only requested columns.
cols = ['timestamp', 'cmdb_id', 'message']
if 'apache_error_messages_window' in globals():
    # Ensure requested columns exist
    avail_cols = [c for c in cols if c in apache_error_messages_window.columns]
    result = apache_error_messages_window.loc[:, avail_cols].sort_values('timestamp').head(50)
else:
    # Fallback: empty DataFrame with desired columns
    import pandas as pd
    result = pd.DataFrame(columns=cols)

result

```
Out[15]:
```
summary = (
    "Summary of raw error logs in 2021-03-12 04:12:00 → 04:17:30 (UTC+8):\n"
    "- Total rows: 19 (apache01 = 10, apache02 = 9).\n"
    "- Time range in this interval: earliest 2021-03-12 04:13:26 (apache02), latest 2021-03-12 04:17:01 (apache01).\n"
    "- All messages are HTTP POST-related entries targeting /UOCP/... ServiceTest endpoints (mix of /person/ and /base/ paths) on apache01/apache02.\n"
    "- Messages appear to be request traces (truncated in display); no explicit HTTP status codes are visible in the shown text—inspect full message text or access logs to extract status codes and detailed error payloads.\n"
    "- Suggested next steps: review full error message texts and web server access logs for HTTP status codes, correlate with Tomcat/backend trace anomalies around 04:16, and check backend service health for the referenced endpoints."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id                                            message
0  2021-03-12 04:13:26+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json...
1  2021-03-12 04:13:33+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest3.json...
2  2021-03-12 04:13:41+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json...
3  2021-03-12 04:13:41+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest11.json ...
4  2021-03-12 04:13:50+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H...
5  2021-03-12 04:14:11+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest5.json...
6  2021-03-12 04:14:12+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H...
7  2021-03-12 04:14:19+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest2.json H...
8  2021-03-12 04:14:19+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...
9  2021-03-12 04:14:34+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H...
10 2021-03-12 04:15:36+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest2.json H...
11 2021-03-12 04:15:54+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest1.json...
12 2021-03-12 04:15:58+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest4.json H...
13 2021-03-12 04:16:07+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...
14 2021-03-12 04:16:29+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...
15 2021-03-12 04:16:30+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H...
16 2021-03-12 04:16:38+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H...
17 2021-03-12 04:16:45+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H...
18 2021-03-12 04:17:01+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H...```
```

In [ ]:
# In[16]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_log if present, otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure timestamp is datetime and localized to UTC+8
if not pd.api.types.is_datetime64_any_dtype(df_log['timestamp']):
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s')
if df_log['timestamp'].dt.tz is None:
    df_log['timestamp'] = df_log['timestamp'].dt.tz_localize(tz)
else:
    df_log['timestamp'] = df_log['timestamp'].dt.tz_convert(tz)

# Define interval (inclusive) in UTC+8
start = pd.Timestamp("2021-03-12 04:12:00", tz=tz)
end = pd.Timestamp("2021-03-12 04:17:00", tz=tz)

# Filter rows and store full filtered set for later use
apache_minute_logs_window = df_log[
    (df_log['timestamp'] >= start) &
    (df_log['timestamp'] <= end) &
    (df_log['cmdb_id'].isin(['apache01', 'apache02'])) &
    (df_log['log_name'].isin(['log.error_count', 'log.row_count']))
].copy()

# Prepare raw rows view (limited to 50) sorted by timestamp asc then cmdb_id
raw_rows_view = apache_minute_logs_window.sort_values(['timestamp', 'cmdb_id']).loc[:, ['timestamp', 'cmdb_id', 'log_name', 'value']].head(50).reset_index(drop=True)

# Compute aggregate per cmdb_id and log_name:
# - earliest timestamp in interval where value > 0 (or NaT if none)
# - maximum value in the interval
grp_all = apache_minute_logs_window.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
    window_count=('timestamp', 'count'),
    max_value_in_interval=('value', 'max')
)

# Earliest timestamp where value > 0
mask_pos = apache_minute_logs_window['value'] > 0
grp_earliest_pos = (
    apache_minute_logs_window[mask_pos]
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(earliest_positive_timestamp=('timestamp', 'min'))
)

agg_table = grp_all.merge(grp_earliest_pos, on=['cmdb_id', 'log_name'], how='left')

# Sort aggregate for compact display: earliest_positive_timestamp asc (NaT last), then max_value desc
agg_table = agg_table.sort_values(by=['earliest_positive_timestamp', 'max_value_in_interval'], ascending=[True, False]).reset_index(drop=True)

# Display the limited raw rows and the aggregate summary (compact)
raw_rows_view, agg_table

```
Out[16]:
```
summary = (
    "Summary of apache logs (apache01 & apache02) between 2021-03-12 04:12:00 and 04:17:00+08:00:\n"
    "- Raw rows returned: 24 rows (per-minute log.row_count and log.error_count for both hosts).\n"
    "- Aggregate by series:\n"
    "  * apache01, log.row_count: 6 points, max value = 675.0, first positive row_count at 2021-03-12 04:12:00+08:00.\n"
    "  * apache02, log.row_count: 6 points, max value = 672.0, first positive row_count at 2021-03-12 04:12:00+08:00.\n"
    "  * apache01, log.error_count: 6 points, max value = 3.0, earliest error (value>0) at 2021-03-12 04:13:00+08:00.\n"
    "  * apache02, log.error_count: 6 points, max value = 4.0, earliest error (value>0) at 2021-03-12 04:13:00+08:00.\n"
    "- Interpretation: request volumes (row_count) are steady in the 600s, and error_count begins at 04:13:00 on both apache01 and apache02 and continues through the interval. "
    "Correlation with backend traces and error messages around 04:13–04:17 is recommended to identify root cause."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp   cmdb_id         log_name  value
0  2021-03-12 04:12:00+08:00  apache01  log.error_count    0.0
1  2021-03-12 04:12:00+08:00  apache01    log.row_count  580.0
2  2021-03-12 04:12:00+08:00  apache02  log.error_count    0.0
3  2021-03-12 04:12:00+08:00  apache02    log.row_count  635.0
4  2021-03-12 04:13:00+08:00  apache01  log.error_count    1.0
..                       ...       ...              ...    ...
19 2021-03-12 04:16:00+08:00  apache02    log.row_count  643.0
20 2021-03-12 04:17:00+08:00  apache01  log.error_count    1.0
21 2021-03-12 04:17:00+08:00  apache01    log.row_count  656.0
22 2021-03-12 04:17:00+08:00  apache02  log.error_count    0.0
23 2021-03-12 04:17:00+08:00  apache02    log.row_count  640.0

[24 rows x 4 columns],     cmdb_id         log_name  window_count  max_value_in_interval earliest_positive_timestamp
0  apache01    log.row_count             6                  675.0   2021-03-12 04:12:00+08:00
1  apache02    log.row_count             6                  672.0   2021-03-12 04:12:00+08:00
2  apache02  log.error_count             6                    4.0   2021-03-12 04:13:00+08:00
3  apache01  log.error_count             6                    3.0   2021-03-12 04:13:00+08:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```